# 01. La plantilla y los jugadores

## Mi pregunta

Quiero saber con que jugadores cuenta de verdad Ecuador en noviembre de 2026, y que forma tienen en sus clubes, antes de hablar de sistemas o de tactica.

## Por que importa

Un sistema no se elige por el nombre del entrenador que lo uso. Se elige por los pies que lo van a jugar. Si mis centrales no tienen minutos, si mi delantero no marca en su club, o si mi portero esta en su mejor momento a los 39 anos, eso cambia la decision.

En este cuaderno cargo la convocatoria de noviembre de 2026, reviso edades, partidos con la seleccion, goles, minutos en el club y la forma reciente. Todo con la fuente y la temporada al lado de cada numero.

In [1]:
import pathlib
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11
BASE = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DATA = BASE / "data"
CHARTS = BASE / "charts"
CHARTS.mkdir(exist_ok=True)
print("Base:", BASE)
print("Charts:", CHARTS)

jug = pd.read_csv(DATA / "jugadores_noviembre_2026.csv")
print(jug.shape)
print(jug[["jugador", "posicion", "club", "edad", "partidos_seleccion", "goles_seleccion"]].to_string(index=False))

Base: /home/hatch/workspace/ecuador-analytics
Charts: /home/hatch/workspace/ecuador-analytics/charts
(30, 13)
             jugador                            posicion                                  club  edad  partidos_seleccion  goles_seleccion
     Hernan Galindez                             Portero                               Huracan    39                  43                0
      Moises Ramirez                             Portero                            AE Kifisia    26                   7                0
       Gonzalo Valle                             Portero                             LDU Quito    30                   5                0
       Willian Pacho           Defensa central izquierdo                   Paris Saint-Germain    25                  43                2
      Piero Hincapie Defensa central / lateral izquierdo                               Arsenal    24                  58                3
        Felix Torres             Defensa central derecho Inter

In [2]:
resumen = jug.groupby("grupo").agg(jugadores=("jugador", "count"), edad_media=("edad", "mean"), partidos_seleccion=("partidos_seleccion", "sum"), goles_seleccion=("goles_seleccion", "sum")).round(1)
print(resumen)
print()
print("Total de filas con nombre en el archivo:", len(jug))
print("Total de partidos con la seleccion sumados:", int(jug["partidos_seleccion"].sum()))
print("Total de goles con la seleccion sumados:", int(jug["goles_seleccion"].sum()))
print("Edad media de la plantilla:", round(jug["edad"].mean(), 1))
print("Jugador mas joven:", jug.loc[jug["edad"].idxmin(), "jugador"], int(jug["edad"].min()), "anos")
print("Jugador mayor:", jug.loc[jug["edad"].idxmax(), "jugador"], int(jug["edad"].max()), "anos")

             jugadores  edad_media  partidos_seleccion  goles_seleccion
grupo                                                                  
Ataque              12        23.3                 283               31
Centrocampo          7        24.3                 180                6
Defensa              8        26.0                 312               16
Portero              3        31.7                  55                0

Total de filas con nombre en el archivo: 30
Total de partidos con la seleccion sumados: 830
Total de goles con la seleccion sumados: 53
Edad media de la plantilla: 25.1
Jugador mas joven: Juan Riquelme Angulo 18 anos
Jugador mayor: Hernan Galindez 39 anos


In [3]:
top = jug.sort_values("partidos_seleccion", ascending=True).tail(15)
plt.figure()
plt.barh(top["jugador"], top["partidos_seleccion"], color="#1f4e79")
plt.xlabel("Partidos con la seleccion, a 17 nov 2026, fuente Wikipedia")
plt.title("Experiencia internacional: los 15 jugadores con mas partidos")
plt.tight_layout()
plt.savefig(CHARTS / "01_partidos_seleccion_top15.png", dpi=150)
plt.show()

In [4]:
plt.figure()
for grupo in jug["grupo"].unique():
    sub = jug[jug["grupo"] == grupo]
    plt.scatter(sub["edad"], sub["partidos_seleccion"], label=grupo, s=90)
    for _, r in sub.iterrows():
        plt.annotate(r["jugador"].split()[-1], (r["edad"], r["partidos_seleccion"]), fontsize=7)
plt.xlabel("Edad a 18 nov 2026")
plt.ylabel("Partidos con la seleccion")
plt.title("Edad y experiencia: no es una plantilla joven sin rodaje")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(CHARTS / "02_edad_y_experiencia.png", dpi=150)
plt.show()

In [5]:
club = jug.dropna(subset=["goles_club"]).copy()
club = club.sort_values("goles_club", ascending=False)
print(club[["jugador", "club_temporada_referencia", "partidos_club", "minutos_club", "goles_club", "asistencias_club"]].to_string(index=False))
print()
print("Jugadores sin datos de club verificados en el contexto:", int(jug["goles_club"].isna().sum()), "de", len(jug))
print("Eso es un limite, no un vacio que voy a rellenar con estimaciones.")

         jugador                  club_temporada_referencia  partidos_club  minutos_club  goles_club asistencias_club
   Jordy Caicedo                          Huracan liga 2026           29.0        2506.0        14.0              NaN
 Kevin Rodriguez                      Union SG liga 2025-26           31.0        1825.0        10.0                2
    John Mercado                        Sparta liga 2025-26           26.0           NaN         7.0                4
   Gonzalo Plata                      Flamengo Serie A 2025           28.0        1346.0         4.0                1
  Moises Caicedo             Chelsea Premier League 2025-26           33.0        2795.0         3.0                1
      Pedro Vite                      Pumas Liga MX 2025-26           38.0        2897.0         3.0                1
   Willian Pacho                          PSG total 2025-26           47.0        4108.0         2.0                1
  Piero Hincapie                      Arsenal total 2025

## Lo que encontre

La plantilla tiene experiencia real. Moises Caicedo suma 67 partidos con la seleccion a los 25 anos, Angelo Preciado 63, Pervis Estupinan 60, Piero Hincapie 58, Gonzalo Plata 55 y Felix Torres 53, todos a 17 de noviembre de 2026, segun Wikipedia. No es un grupo que necesite aprender a jugar juntos desde cero.

La edad media esta cerca de los 26 anos, con extremos claros. Juan Riquelme Angulo tiene 18 anos y 4 partidos, Kendry Paez tiene 19 anos y 28 partidos, y Hernan Galindez tiene 39 anos y 43 partidos. Galindez, ademas, llega en forma en su club. En Huracan, en 2026, jugo 32 partidos, con 11 porterias a cero y un 74.0 por ciento de paradas, segun Flashscore a 14 de noviembre de 2026.

El delantero en mejor forma es Jordy Caicedo. En la liga de Huracan en 2026 marco 14 goles en 29 partidos y 2.506 minutos. Kevin Rodriguez marco 10 goles en 31 partidos de liga con Union Saint-Gilloise en 2025-26. Esa diferencia de forma es un dato, no una opinion, y pesa en la eleccion del delantero titular.

Hay dos avisos que no voy a esconder. Primero, Piero Hincapie solo jugo 67 minutos en la Premier League en 2026-27, con 0 titularidades en liga, aunque su total en el Arsenal en 2025-26 fue de 39 partidos y 2.763 minutos. Segundo, Joel Ordonez tiene 0 minutos en 2026-27 por una fractura del quinto metatarsiano, con segunda cirugia el 25 de septiembre de 2026 y regreso previsto en enero de 2027. Eso deja la defensa central corta para noviembre.

## Lo que significa

Ecuador tiene un nucleo con partidos, un portero en forma, un delantero en racha y dos centrales de elite en Pacho y Hincapie, pero con minutos recientes muy distintos. Cualquier sistema que proponga tiene que proteger esa defensa central, no exponerla, y tiene que poner a Jordy Caicedo cerca del area, donde esta marcando.